In [1]:
import uproot
import awkward as ak
import dask_awkward as dak
import numpy as np
from dask.diagnostics import ProgressBar

In [2]:
# ==============================================================================
# FUNÇÃO DE TRUTH MATCHING
# ==============================================================================
def apply_strict_truth_match(pdg, mother, **branches):
    local_idx = ak.local_index(pdg)

    safe_mom = ak.where(mother >= 0, mother, 0)
    mom_pdg  = pdg[safe_mom]
    mom_pdg  = ak.where(mother >= 0, mom_pdg, -999)

    safe_grandmom = safe_mom[safe_mom]
    grandmom_pdg  = pdg[safe_grandmom]
    grandmom_pdg  = ak.where(safe_mom >= 0, grandmom_pdg, -999)

    B_ancestor_idx = ak.full_like(pdg, -1)
    B_ancestor_idx = ak.where((pdg == 511), local_idx, B_ancestor_idx)
    B_ancestor_idx = ak.where((pdg == 313) & (mom_pdg == 511), safe_mom, B_ancestor_idx)
    B_ancestor_idx = ak.where((pdg == 22)  & (mom_pdg == 511), safe_mom, B_ancestor_idx)

    is_K    = (pdg == 321)  & (mom_pdg == 313) & (grandmom_pdg == 511)
    is_pi   = (pdg == -211) & (mom_pdg == 313) & (grandmom_pdg == 511)
    is_mu_m = (pdg == 13)   & (mom_pdg == 22)  & (grandmom_pdg == 511)
    is_mu_p = (pdg == -13)  & (mom_pdg == 22)  & (grandmom_pdg == 511)

    B_ancestor_idx = ak.where(is_K,               safe_grandmom, B_ancestor_idx)
    B_ancestor_idx = ak.where(is_pi,              safe_grandmom, B_ancestor_idx)
    B_ancestor_idx = ak.where(is_mu_m | is_mu_p,  safe_grandmom, B_ancestor_idx)

    anc_kstar = B_ancestor_idx[(pdg == 313) & (mom_pdg == 511)]
    anc_K     = B_ancestor_idx[is_K]
    anc_pi    = B_ancestor_idx[is_pi]
    anc_mu_m  = B_ancestor_idx[is_mu_m]
    anc_mu_p  = B_ancestor_idx[is_mu_p]

    p_anc = B_ancestor_idx[:, :, np.newaxis]

    has_1_kstar = ak.sum(p_anc == anc_kstar[:, np.newaxis, :], axis=-1) == 1
    has_1_K     = ak.sum(p_anc == anc_K[:, np.newaxis, :],     axis=-1) == 1
    has_1_pi    = ak.sum(p_anc == anc_pi[:, np.newaxis, :],    axis=-1) == 1
    has_1_mu_m  = ak.sum(p_anc == anc_mu_m[:, np.newaxis, :],  axis=-1) == 1
    has_1_mu_p  = ak.sum(p_anc == anc_mu_p[:, np.newaxis, :],  axis=-1) == 1

    is_valid_chain  = has_1_kstar & has_1_K & has_1_pi & has_1_mu_m & has_1_mu_p
    particle_mask   = is_valid_chain & (B_ancestor_idx >= 0)
    event_has_signal = ak.sum(particle_mask, axis=1) > 0

    results = {'pdg': pdg[particle_mask][event_has_signal]}
    for name, array in branches.items():
        results[name] = array[particle_mask][event_has_signal]

    return results

In [3]:
# ==============================================================================
# WRAPPER POR PARTIÇÃO
# ==============================================================================
def process_partition(pdg_part, mother_part, mass_part):
    result = apply_strict_truth_match(
        pdg    = pdg_part,
        mother = mother_part,
        mass   = mass_part
    )
    return ak.Array({'pdg': result['pdg'], 'mass': result['mass']})

In [4]:
file_path = "/eos/user/t/tdeandra/BPH_Gen_NoFilter.root"
print("Abrindo arquivo e montando grafo lazy...")

Abrindo arquivo e montando grafo lazy...


In [5]:
arrays = uproot.dask(
    {file_path: "Events"},
    expressions=["BPHGenPart_pdgId", "BPHGenPart_genPartIdxMother", "BPHGenPart_mass"],
    library="ak"
)

In [6]:
raw_pdg    = arrays["BPHGenPart_pdgId"]
raw_mother = arrays["BPHGenPart_genPartIdxMother"]
raw_mass   = arrays["BPHGenPart_mass"]

In [7]:
result_lazy = dak.map_partitions(
    process_partition,
    raw_pdg,
    raw_mother,
    raw_mass
)

Número de partições: 1


In [8]:
with ProgressBar():
    result = result_lazy.compute()

[########################################] | 100% Completed | 102.38 ms
[########################################] | 100% Completed | 68.09 s


In [9]:
pdg_clean  = result["pdg"]
mass_clean = result["mass"]

print(f"\nTotal de eventos de SINAL retidos: {len(pdg_clean)}")

for i in range(min(5, len(pdg_clean))):
    print(f"\nEvento {i}:")
    print(f"  PDG  : {pdg_clean[i].tolist()}")
    print(f"  Mass : {mass_clean[i].tolist()}")


Total de eventos de SINAL retidos: 1631702

Evento 0:
  PDG  : [511, 313, -13, 22, 13, 22, 22, 22, 321, -211]
  Mass : [5.28125, 0.90625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.494140625, 0.0]

Evento 1:
  PDG  : [511, 313, -13, 22, 13, 22, 321, -211]
  Mass : [5.28125, 0.892578125, 0.0, 0.0, 0.0, 0.0, 0.494140625, 0.0]

Evento 2:
  PDG  : [511, 313, -13, 22, 13, 22, 22, 321, -211]
  Mass : [5.28125, 0.896484375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.494140625, 0.0]

Evento 3:
  PDG  : [511, 313, -13, 22, 13, 22, 22, 22, 22, 321, -211]
  Mass : [5.28125, 0.951171875, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.494140625, 0.0]

Evento 4:
  PDG  : [511, 313, -13, 22, 13, 22, 321, -211]
  Mass : [5.28125, 0.923828125, 0.0, 0.0, 0.0, 0.0, 0.494140625, 0.0]
